# Batch Processing for Multiple LIF Containers (Co-culture)

This notebook runs dual-channel cell segmentation, A549/HPFa colocalization cleaning, `%_area_cleaned` filtering, and per-cell feature extraction in batch mode for all or selected `.lif` containers.

Images whose filename contains `5x` are skipped.

CSV outputs are written to `../coc_results/<lif_container_id>/<lif_image_name>.csv`.
Original Cellpose cell-label TIFFs are written under `RAW_DATA_DIRECTORY/cell_labels/<lif_container_id>/`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.batch_processing import build_coc_runtime_config, run_coc_batch

pd.set_option("display.max_columns", None)



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


creating new log file
2026-10-12 15:43:10,794 [INFO] WRITING LOG OUTPUT TO C:\Users\adiez_cmic\.cellpose\run.log
2026-10-12 15:43:10,794 [INFO] 
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0


In [2]:
# Paths
RAW_DATA_DIRECTORY = r"Z:\margrethe.c.stahl\Co-culture"
RESULTS_ROOT = Path("../coc_results")

# Marker configuration: (marker_name, channel_index)
MARKERS = (
    ("A549_green", 0),
    ("HPFa_red", 1),
    ("brightfield", 2),
)

# Cell segmentation settings
CELL_CHANNELS = (0, 1)
CELL_LABELING = "cellpose"  # "cellpose" or "voronoi-otsu"
SPOT_SIGMA = 2
OUTLINE_SIGMA = 2
MIN_MAX_CELL_VOLUME = None  # e.g. (500, 6500)
PCT_AREA_CLEANED_THRESHOLD = 20

# Batch behavior
CONTAINER_INDICES: list[int] | None = None  # e.g. [0, 2, 3]
IMAGE_INDICES: list[int] | None = None      # e.g. [0, 1, 5]
OVERWRITE_CSV = False

In [3]:
NOTEBOOK_CONFIG = {
    "results_root": str(RESULTS_ROOT),
    "markers": [
        {"name": name, "channel": channel}
        for name, channel in MARKERS
    ],
    "cell_channels": list(CELL_CHANNELS),
    "cell_labeling": CELL_LABELING,
    "spot_sigma": SPOT_SIGMA,
    "outline_sigma": OUTLINE_SIGMA,
    "min_max_cell_volume": (
        list(MIN_MAX_CELL_VOLUME)
        if MIN_MAX_CELL_VOLUME is not None
        else None
    ),
    "pct_area_cleaned_threshold": PCT_AREA_CLEANED_THRESHOLD,
    "container_indices": CONTAINER_INDICES,
    "image_indices": IMAGE_INDICES,
    "overwrite_csv": OVERWRITE_CSV,
}

CONFIG = build_coc_runtime_config(
    user_config=NOTEBOOK_CONFIG,
    raw_data_directory=RAW_DATA_DIRECTORY,
)
CONFIG

{'raw_data_directory': WindowsPath('Z:/margrethe.c.stahl/Co-culture'),
 'results_root': WindowsPath('../coc_results'),
 'markers': (('A549_green', 0), ('HPFa_red', 1), ('brightfield', 2)),
 'cell_channels': (0, 1),
 'cell_labeling': 'cellpose',
 'spot_sigma': 2.0,
 'outline_sigma': 2.0,
 'min_max_cell_volume': None,
 'pct_area_cleaned_threshold': 20.0,
 'container_indices': None,
 'image_indices': None,
 'overwrite_csv': False}

In [4]:
batch_results = run_coc_batch(CONFIG)
batch_results

[2026-10-12 15:43:10] | INFO | container=Co-culture-day1-A549-24hpriortoHPFa | image=all | stage=container_start | msg=starting COC container batch | elapsed_s=na | details={"images_selected": 10, "images_total": 10}
[2026-10-12 15:43:10] | INFO | container=Co-culture-day1-A549-24hpriortoHPFa | image=image_0 | stage=image_start | msg=starting COC image processing | elapsed_s=0.00 | details=na
[2026-10-12 15:43:10] | WARN | container=Co-culture-day1-A549-24hpriortoHPFa | image=D1DS1+FGF-2.3-5x | stage=skip | msg=filename contains '5x' | elapsed_s=0.00 | details={"image_index": 0}
[2026-10-12 15:43:10] | INFO | container=Co-culture-day1-A549-24hpriortoHPFa | image=image_1 | stage=image_start | msg=starting COC image processing | elapsed_s=0.00 | details=na
[2026-10-12 15:43:10] | WARN | container=Co-culture-day1-A549-24hpriortoHPFa | image=D1DS1-FGF-2.3-5x | stage=skip | msg=filename contains '5x' | elapsed_s=0.00 | details={"image_index": 1}
[2026-10-12 15:43:10] | INFO | container=Co-c

{'containers': 4,
 'images_total': 39,
 'processed': 39,
 'success': 23,
 'skipped': 16,
 'failed': 0,
 'elapsed_min': 90.82912865333492,
 'failures': []}